# Salesforce Object Ingestion

Pulls a single Salesforce object's field metadata via the REST describe API
and writes it to a Fabric Lakehouse as a Delta table. Use as a starting point
for building a Salesforce data dictionary alongside other ingestion patterns
(SOQL -> Delta, bulk exports, etc.).

**Usage:**
1. Set the Azure Key Vault URL and the three Salesforce credential secret names below.
2. Set `SALESFORCE_OBJECT` to the API name of the Salesforce object to describe
   (e.g. `Account`, `Opportunity`, `MyCustomObject__c`).
3. Set `TARGET_SCHEMA` and `TARGET_TABLE` to the destination in the attached Lakehouse.
4. Run all cells.

**Auth:** This notebook uses the classic username + password + security-token flow
via `simple_salesforce`. For production use, consider migrating to an OAuth 2.0
JWT bearer flow via a Salesforce Connected App - it avoids rotating security
tokens and scales better to service accounts.

## Configuration

Secret names below are the **keys in Azure Key Vault**, not the secret values.
The notebook reads them at runtime via `notebookutils.credentials.getSecret`, which
uses the workspace identity - no SPN certificate or secret lives in the notebook.

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────────
# Azure Key Vault that stores the Salesforce credentials.
AZURE_KEY_VAULT_URL = "<AzureKeyVaultUrl>"

# Names of the three secrets in Key Vault. Values of each:
#   - Username secret: Salesforce login email (e.g. "integration@example.com")
#   - Password secret: Salesforce login password
#   - Token secret:    Salesforce user security token (reset under My Settings -> Reset My Security Token)
SALESFORCE_USERNAME_SECRET_NAME = "<SalesforceUsernameSecretName>"
SALESFORCE_PASSWORD_SECRET_NAME = "<SalesforcePasswordSecretName>"
SALESFORCE_TOKEN_SECRET_NAME    = "<SalesforceTokenSecretName>"

# Salesforce object API name. Custom objects end in "__c".
SALESFORCE_OBJECT = "<SalesforceObjectName>"

# Target Lakehouse schema + table. Schema-enabled Lakehouse required for the
# two-part name; for non-schema Lakehouses use "dbo" as the schema.
TARGET_SCHEMA = "<LakehouseSchemaName>"
TARGET_TABLE  = "<LakehouseTableName>"

print("--- Configuration ---")
print(f"  Key Vault:        {AZURE_KEY_VAULT_URL}")
print(f"  Salesforce object: {SALESFORCE_OBJECT}")
print(f"  Target table:     {TARGET_SCHEMA}.{TARGET_TABLE}")

## Optional: resolve configuration from a Variable Library

Set `VARIABLE_LIBRARY_NAME` to a Variable Library in this workspace and any configuration
value above still holding its `<Placeholder>` (or blank) resolves from the library's
**active value set** - one notebook, per-workspace config. Leave it blank to skip.

Limits: same-workspace libraries only, and `notebookutils.variableLibrary` has **no
service-principal support** - SPN / scheduled runs should pass real values in the
configuration cell (or as notebook parameters) instead.

In [ ]:
# ── Optional Variable Library resolution ─────────────────────────────────────────
# Blank = skip; the literals in the configuration cell are used as-is.
VARIABLE_LIBRARY_NAME = ""


def vl_lookup(variable_name: str) -> str:
    """
    Resolve one Variable Library variable from the workspace's active value set.

    :param variable_name: Variable name in the library (case-sensitive).
    :returns: The variable's value.
    :raises RuntimeError: If the library or variable cannot be resolved here.
    """
    try:
        return notebookutils.variableLibrary.get(
            f"$(/**/{VARIABLE_LIBRARY_NAME}/{variable_name})"
        )
    except Exception as exc:
        raise RuntimeError(
            f"Variable Library lookup failed for '{variable_name}' - is "
            f"'{VARIABLE_LIBRARY_NAME}' in this workspace, and does this runtime support "
            "notebookutils.variableLibrary?"
        ) from exc


def resolve(current: str, variable_name: str) -> str:
    """
    Keep an explicitly-set value; resolve blank or <Placeholder> values from the library.

    :param current: The configuration value as currently set.
    :param variable_name: Variable Library variable to fall back to.
    :returns: The effective configuration value.
    """
    if current and not (current.startswith("<") and current.endswith(">")):
        return current
    return vl_lookup(variable_name)


if VARIABLE_LIBRARY_NAME:
    AZURE_KEY_VAULT_URL = resolve(AZURE_KEY_VAULT_URL, "AzureKeyVaultUrl")
    SALESFORCE_USERNAME_SECRET_NAME = resolve(
        SALESFORCE_USERNAME_SECRET_NAME, "SalesforceUsernameSecretName"
    )
    SALESFORCE_PASSWORD_SECRET_NAME = resolve(
        SALESFORCE_PASSWORD_SECRET_NAME, "SalesforcePasswordSecretName"
    )
    SALESFORCE_TOKEN_SECRET_NAME = resolve(
        SALESFORCE_TOKEN_SECRET_NAME, "SalesforceTokenSecretName"
    )
    SALESFORCE_OBJECT = resolve(SALESFORCE_OBJECT, "SalesforceObjectName")
    TARGET_SCHEMA = resolve(TARGET_SCHEMA, "LakehouseSchemaName")
    TARGET_TABLE = resolve(TARGET_TABLE, "LakehouseTableName")
    print(f"Configuration resolved via Variable Library '{VARIABLE_LIBRARY_NAME}'.")

## Authentication

Installs `simple_salesforce`, pulls credentials from Key Vault, and builds a
client. The client is reused by all downstream cells.

In [ ]:
%pip install simple_salesforce

In [ ]:
from simple_salesforce import Salesforce


def create_salesforce_client() -> Salesforce:
    """
    Build a simple_salesforce client using credentials stored in Azure Key Vault.

    Credentials are read via notebookutils.credentials.getSecret, which uses the
    workspace identity - the notebook never sees the raw secret values in code.

    :returns: Authenticated simple_salesforce.Salesforce client.
    :raises simple_salesforce.exceptions.SalesforceAuthenticationFailed: If
        credentials are invalid or the security token is stale (Salesforce
        requires a new token after password changes or IP allow-list changes).
    """
    username = notebookutils.credentials.getSecret(
        AZURE_KEY_VAULT_URL, SALESFORCE_USERNAME_SECRET_NAME
    )
    password = notebookutils.credentials.getSecret(
        AZURE_KEY_VAULT_URL, SALESFORCE_PASSWORD_SECRET_NAME
    )
    security_token = notebookutils.credentials.getSecret(
        AZURE_KEY_VAULT_URL, SALESFORCE_TOKEN_SECRET_NAME
    )
    return Salesforce(
        username=username,
        password=password,
        security_token=security_token,
    )


sf = create_salesforce_client()
print(f"Connected to Salesforce instance: {sf.sf_instance}")

## Extract Field Metadata

Calls `describe()` on the configured Salesforce object, flattens the field list
into a single-row-per-field shape, and writes to the target Lakehouse as a
Delta table with `overwriteSchema = true`. Re-running replaces the table
contents - safe to schedule.

All columns are coerced to `string` so the table shape is stable across
objects (some Salesforce field types omit `length` / `precision` / `scale`).
If you need natively-typed columns for a specific object, drop the cast loop.

In [ ]:
import pandas as pd
from pyspark.sql.types import StringType
from pyspark.sql.functions import col, coalesce, lit


def extract_field_metadata(sf: Salesforce, object_name: str) -> pd.DataFrame:
    """
    Describe a Salesforce object and return its field list as a DataFrame.

    Object-level properties (child relationships, record types) are captured
    once from the describe payload and attached to each field row so the
    output table is self-contained per object.

    :param sf: Authenticated simple_salesforce client.
    :param object_name: Salesforce object API name (e.g. "Account", "Opportunity",
        "CustomObject__c"). Must be visible to the authenticating user.
    :returns: DataFrame with one row per field. Column set is fixed regardless
        of which object is described.
    :raises simple_salesforce.exceptions.SalesforceResourceNotFound: If the
        object does not exist in the target org.
    """
    metadata = getattr(sf, object_name).describe()

    child_rels = ", ".join(
        r["childSObject"] for r in (metadata.get("childRelationships") or [])
    )
    record_types = ", ".join(
        rt["name"] for rt in (metadata.get("recordTypeInfos") or [])
    )

    rows = []
    for field in metadata["fields"]:
        picklist_values = ""
        if field["type"] == "picklist":
            picklist_values = ", ".join(
                p["value"] for p in field.get("picklistValues", [])
            )

        rows.append({
            "Object_Name":         object_name,
            "Field_Label":         field["label"],
            "API_Name":            field["name"],
            "Data_Type":           field["type"],
            "Length":              field.get("length"),
            "Precision":           field.get("precision"),
            "Scale":               field.get("scale"),
            "Required":            not field["nillable"],
            "Unique":               field.get("unique", False),
            "External_Id":         field.get("externalId", False),
            "Searchable":          field.get("searchable", False),
            "Filterable":          field.get("filterable", False),
            "Picklist_Values":     picklist_values,
            "Default_Value":       field.get("defaultValue"),
            "Description":         field.get("inlineHelpText"),
            "Custom_Field":        field["custom"],
            "Parent_Relationship": field.get("relationshipName"),
            "Child_Relationships": child_rels,
            "Record_Types":        record_types,
        })

    return pd.DataFrame(rows)


field_df = extract_field_metadata(sf, SALESFORCE_OBJECT)
print(f"Extracted {len(field_df)} field(s) from {SALESFORCE_OBJECT}.")

# Convert to Spark and coerce every column to string. Nulls become empty
# strings so downstream SQL filters don't need IS NULL handling. This keeps
# the target table shape stable if the notebook is re-run against a different
# object later.
spark_df = spark.createDataFrame(field_df)
for col_name in spark_df.columns:
    spark_df = spark_df.withColumn(
        col_name, coalesce(col(col_name).cast(StringType()), lit(""))
    )

full_table_name = f"{TARGET_SCHEMA}.{TARGET_TABLE}"
(
    spark_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(full_table_name)
)

print(f"Wrote field metadata for {SALESFORCE_OBJECT} to {full_table_name}.")